# Vision B's final figures on the RSNA test split, opened once (DEC-22, DEC-31 §1 item 3; Phase 4, Kaggle)

Vision B is final: EXP-6's ResNet-50, its top-2 aggregation and Platt calibration, and DEC-28's threshold 0.175 (Item 2 kept it). DEC-22 keeps the RSNA test split closed until then; DEC-31 opens it **once**, for these figures, fixed before any is seen:
- the scan AUC;
- sensitivity and specificity at 0.175, with patient-bootstrap intervals;
- sensitivity per bleed subtype;
- the ECE of the calibrated scan probability.

Slice AUC, Wilson intervals, counts, log loss and the reliability bins are added as labelled extras. Nothing is chosen or tuned. The logic is in `src/strokecopilot/vision/haemorrhage/final_check.py`.

**How the one opening is protected.** The notebook first runs the identical report on validation and checks it against EXP-6 (scan AUC 0.977) and DEC-28 (sensitivity at 0.175 at least 0.955). The test split is opened only if that passes, and only if `OPEN_TEST_SPLIT = True` in section 1. With `False` it is a dry run: validation only, nothing opened.

**Data rules (`docs/DATA_USE.md`).** RSNA data stays on Kaggle. The split holds IDs, so it is rebuilt here and never saved. Output files hold counts, rates and thresholds only; section 9 checks for IDs before saving. Model weights stay private to the team.

**Setup** (as for notebooks 22 and 24)
1. Upload this notebook to Kaggle (File → Import Notebook).
2. Add Input: the four `rsna-slim-v1-file-s0` to `-s3` datasets, and the private weights dataset holding EXP-6's `best.pt`, `calibration.json` and `metrics.json`.
3. Settings: accelerator **GPU T4 x2**, **Internet on**. Only one GPU is used.

**Run 1 (dry run):** leave `OPEN_TEST_SPLIT = False`, Save Version → **Save & Run All**. About 10–15 min, free. It ends with `DRY RUN OK:` and the validation figures.
**Run 2 (the real one, once):** set `OPEN_TEST_SPLIT = True`, Save & Run All. About 15–20 min (estimate). It ends with `OK:`.

**Done when** section 9 prints `OK:`. Then download `config.json`, `metrics.json` and `reliability_test.png` from the version's Output and upload them to `StrokeCopilot/runs/<EXPERIMENT>_visionB_rsna_test/` on Drive. Do not run it again: a second run would be a second look at the test split.

## 1. Settings

In [ ]:
EXPERIMENT = 'EXP-60'          # Phase 4's first Experiments row (EXP-60 onward reserved for Phase 4; confirm before Run 2)
OPEN_TEST_SPLIT = False         # Run 1: False (dry run, validation only). Run 2: True, once.
DATA_ROOT = '/kaggle/input'
LOCAL_DIR = '/kaggle/temp'    # never /kaggle/working: Save Version would keep the shards as output
OUT_ROOT = '/kaggle/working'
EXPECTED = {'readable_slices': 752802, 'scans': 21744, 'patients': 18938, 'val_scans': 2166}
BATCH_SIZE = 128
REPO_REF = 'main'
RUN_DIR = f'{OUT_ROOT}/{EXPERIMENT}_visionB_rsna_test'

## 2. Install the package
Installs the package at the current commit of `REPO_REF`. `ALLOW_RSNA_TEST` must not be set by the environment: only section 7 sets it, after the validation check.

In [ ]:
import json, os, shutil, subprocess, sys, time

started_all = time.perf_counter()
if os.environ.get('ALLOW_RSNA_TEST') == '1':
    raise RuntimeError('ALLOW_RSNA_TEST is already set. Only section 7 may set it; unset it and run again.')
repo = 'https://github.com/parsa-hke/strokecopilot.git'
found = subprocess.run(['git', 'ls-remote', repo, REPO_REF], capture_output=True, text=True)
if found.returncode or not found.stdout.strip():
    raise RuntimeError(('Could not read the repo or REPO_REF. Check the branch name. ' + found.stderr)[-800:])
COMMIT = found.stdout.split()[0]
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'git+{repo}@{COMMIT}'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError(result.stderr[-800:])
del repo, found, result
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'], check=True)
    import timm

import numpy as np
import torch
from strokecopilot.data import rsna, rsna_split
from strokecopilot.vision.haemorrhage import aggregate as agg
from strokecopilot.vision.haemorrhage import operating_point as op
from strokecopilot.vision.haemorrhage import train
from strokecopilot.vision.haemorrhage import final_check as fc, infer

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
print(f"Package at {COMMIT[:7]}; torch {torch.__version__}; timm {timm.__version__}; GPU {gpu or 'none'}.")
if DEVICE.type != 'cuda':
    raise RuntimeError('No GPU. Settings → Accelerator → GPU T4 x2, then Save & Run All again.')

## 3. EXP-6 model and the run record
Finds `best.pt`, `calibration.json` and EXP-6's `metrics.json` in the attached weights dataset, applies DEC-28's threshold, then writes `config.json` before anything is computed.

In [ ]:
import glob

def find(name):
    hits = sorted({h for depth in range(1, 5) for h in glob.glob(os.path.join(DATA_ROOT, *['*'] * depth, name))})
    if len(hits) != 1:
        raise FileNotFoundError(f'Expected one {name} in the attached datasets, found {len(hits)}. Attach the weights dataset (and only one).')
    return hits[0]

BEST, CALIBRATION, EXP6_METRICS = find('best.pt'), find('calibration.json'), find('metrics.json')
net, state = train.load_best(BEST, DEVICE)
calibrator = infer.contract_calibrator(CALIBRATION)   # EXP-6's aggregation and Platt calibration, DEC-28's threshold
op.check_monotonic(calibrator)
exp6 = json.load(open(EXP6_METRICS))
if exp6.get('experiment') != 'EXP-6':
    raise ValueError(f"The attached metrics.json is from {exp6.get('experiment')}, not EXP-6.")
EXP6_FINGERPRINT = exp6['split']['fingerprint']
print(f"Model {state['config']['model']} from epoch {state['epoch']}; aggregation {calibrator.method} k={calibrator.k}, "
      f"{calibrator.kind} calibration, DEC-28 threshold {calibrator.threshold}. EXP-6 split fingerprint {EXP6_FINGERPRINT[:12]}.")

def find_test_scans(obj):
    """EXP-6's recorded test-split scan count, if its metrics.json has one (counts only)."""
    if isinstance(obj, dict):
        if isinstance(obj.get('test'), dict) and isinstance(obj['test'].get('scans'), int):
            return obj['test']['scans']
        for v in obj.values():
            n = find_test_scans(v)
            if n is not None:
                return n
    return None
EXP6_TEST_SCANS = find_test_scans(exp6.get('split', {}))

if os.path.exists(os.path.join(RUN_DIR, 'metrics.json')):
    raise FileExistsError(f'{RUN_DIR} already has metrics.json. The test split is opened once; stop and tell Parsa.')
os.makedirs(RUN_DIR, exist_ok=True)
config = {'experiment': EXPERIMENT, 'task': "Vision B's final figures on the RSNA test split, opened once (DEC-22, DEC-31 §1 item 3)",
          'commit': COMMIT, 'repo_ref': REPO_REF, 'preregistration': 'docs/PHASE4_PREREGISTRATION.md §1 item 3 (DEC-31, approved 19 Nov 2026)',
          'open_test_split': OPEN_TEST_SPLIT, 'model_from': 'EXP-6', 'epoch': state['epoch'], 'calibration': vars(calibrator),
          'threshold': 'DEC-28 (infer.CONTRACT_THRESHOLD)', 'split_seed': rsna_split.SEED, 'split_fingerprint': EXP6_FINGERPRINT,
          'bootstrap': fc.BOOTSTRAP, 'bootstrap_seed': fc.SEED, 'subtypes': list(fc.SUBTYPES), 'ece_bins': agg.ECE_BINS,
          'validation_gate': {'scan_auc': fc.VAL_SCAN_AUC, 'tolerance': fc.VAL_AUC_TOLERANCE, 'sensitivity_floor': fc.VAL_SENSITIVITY_FLOOR},
          'exp6_test_scans': EXP6_TEST_SCANS, 'gpu': gpu}
with open(os.path.join(RUN_DIR, 'config.json'), 'w') as f:
    json.dump(config, f, indent=1, default=str)
print(f"config.json written to {RUN_DIR}. OPEN_TEST_SPLIT = {OPEN_TEST_SPLIT}.")

## 4. RSNA shards
The same steps as notebooks 20, 22 and 24: shards are copied to the local disk and checked against their index.

In [ ]:
started = time.perf_counter()
shards = rsna.find_shards(DATA_ROOT)
index = rsna.load_index(shards)
readers = {}
for shard, (path, _) in shards.items():
    if os.path.isdir(path):
        raise ValueError(f'Shard {shard} is attached as a folder. Attach the rsna-slim-v1-file-s{shard} dataset instead.')
    local_path, _ = rsna.copy_shard(path, LOCAL_DIR)
    readers[shard] = rsna.ShardReader(local_path)
    if readers[shard].ids() != set(index.loc[index['shard'] == shard, 'image_id']):
        raise ValueError(f'Shard {shard}: its images and index differ. Check that dataset.')
SETUP_SECONDS = round(time.perf_counter() - started)
print(f'{len(index):,} readable slices in {len(readers)} shards, set up in {SETUP_SECONDS} s.')
if len(index) != EXPECTED['readable_slices']:
    raise ValueError(f"{len(index):,} readable slices, expected {EXPECTED['readable_slices']:,}.")

## 5. The split, exactly as EXP-6 built it
Rebuilt with its fixed seed; it must match EXP-6's fingerprint. Only the validation rows are taken here. The test rows stay unread until section 7.

In [ ]:
scans = rsna.scan_table(index)
assignment = rsna_split.assign(scans)
FINGERPRINT = rsna_split.fingerprint(assignment)
if FINGERPRINT != EXP6_FINGERPRINT:
    raise ValueError('The rebuilt split differs from EXP-6 (fingerprints differ). Stop and tell Parsa.')
scans = rsna_split.add_split(scans, assignment)
index = index.merge(scans[['series_id', 'split']], on='series_id', validate='many_to_one')
val_items = rsna_split.rows(index, 'val').reset_index(drop=True)
known_ids = set(scans['patient_id']) | set(scans['series_id'])
del assignment, scans
n_val_scans = val_items['series_id'].nunique()
print(f"Validation: {val_items['patient_id'].nunique():,} patients, {n_val_scans:,} scans, {len(val_items):,} slices. Split matches EXP-6.")
if n_val_scans != EXPECTED['val_scans']:
    raise ValueError(f"{n_val_scans} validation scans, expected {EXPECTED['val_scans']}.")

## 6. Validation pass: the same report, where the answer is known
EXP-6's `best.pt` on every validation slice, then `final_check.report` exactly as it will run on the test split. The test split is opened only if this passes.

In [ ]:
WORKERS = max(1, os.cpu_count() or 2)
COLS = ['series_id', 'patient_id', 'any', *fc.SUBTYPES, 'prob']
started = time.perf_counter()
val_items = val_items.assign(prob=train.predict(net, train.SliceDataset(val_items, readers), DEVICE, BATCH_SIZE, WORKERS))
VAL_PREDICT_SECONDS = round(time.perf_counter() - started)
VAL_REPORT = fc.report(val_items[COLS], calibrator)
known_ids |= set(val_items['image_id'])
del val_items
print(f'Validation, {VAL_PREDICT_SECONDS} s to predict:')
print('\n'.join(fc.summary_lines(VAL_REPORT)))
PROBLEMS = fc.validation_check(VAL_REPORT)
if PROBLEMS:
    raise ValueError('Validation check failed; the test split stays closed. ' + '; '.join(PROBLEMS) + '. Tell Parsa.')
print('Validation check passed: the pipeline reproduces EXP-6 and DEC-28.')

## 7. The test split, opened once
Only with `OPEN_TEST_SPLIT = True` and after section 6 passed. `ALLOW_RSNA_TEST` is set here and removed straight after the rows are taken.

In [ ]:
TEST_REPORT, TEST_PREDICT_SECONDS = None, None
if not OPEN_TEST_SPLIT:
    print('Dry run: the test split was not opened.')
else:
    os.environ['ALLOW_RSNA_TEST'] = '1'
    try:
        test_items = rsna_split.rows(index, 'test').reset_index(drop=True)
    finally:
        del os.environ['ALLOW_RSNA_TEST']
    n_test_scans = test_items['series_id'].nunique()
    if EXP6_TEST_SCANS is not None and n_test_scans != EXP6_TEST_SCANS:
        raise ValueError(f'{n_test_scans} test scans, but EXP-6 recorded {EXP6_TEST_SCANS}. Stop and tell Parsa.')
    started = time.perf_counter()
    test_items = test_items.assign(prob=train.predict(net, train.SliceDataset(test_items, readers), DEVICE, BATCH_SIZE, WORKERS))
    TEST_PREDICT_SECONDS = round(time.perf_counter() - started)
    TEST_REPORT = fc.report(test_items[COLS], calibrator)
    known_ids |= set(test_items['image_id'])
    del test_items
    print(f'TEST SPLIT ({TEST_PREDICT_SECONDS} s to predict):')
    print('\n'.join(fc.summary_lines(TEST_REPORT)))
del index

## 8. Reliability figure
The test split's reliability bins (counts and rates only, no images or IDs).

In [ ]:
if TEST_REPORT is not None:
    agg.reliability_plot(TEST_REPORT['extras']['reliability'], os.path.join(RUN_DIR, 'reliability_test.png'),
                         title='Vision B scan calibration, RSNA test split (EXP-6, DEC-28)')
    print('reliability_test.png written.')

## 9. Save and check
`metrics.json` goes in the output folder with `config.json`. Nothing is saved if an ID reaches a file.

In [ ]:
metrics = {'experiment': EXPERIMENT, 'commit': COMMIT, 'split_fingerprint': FINGERPRINT, 'open_test_split': OPEN_TEST_SPLIT,
           'validation': VAL_REPORT, 'test': TEST_REPORT,
           'seconds': {'setup': SETUP_SECONDS, 'predict_val': VAL_PREDICT_SECONDS, 'predict_test': TEST_PREDICT_SECONDS},
           'minutes_total': round((time.perf_counter() - started_all) / 60, 1)}
name = 'metrics.json' if OPEN_TEST_SPLIT else 'metrics_dry_run.json'
texts = {name: json.dumps(metrics, indent=1, default=str), 'config.json': open(os.path.join(RUN_DIR, 'config.json')).read()}
for fname, text in texts.items():
    if fc.ids_in(text, known_ids):
        raise ValueError(f'An ID reached {fname}. Nothing was saved; stop and check.')
for fname, text in texts.items():
    with open(os.path.join(RUN_DIR, fname), 'w') as f:
        f.write(text)
for reader in readers.values():
    reader.close()
shutil.rmtree(LOCAL_DIR, ignore_errors=True)
del known_ids
print('Saved:', sorted(os.listdir(RUN_DIR)))
v = VAL_REPORT['preregistered']
if not OPEN_TEST_SPLIT:
    print(f"DRY RUN OK: {EXPERIMENT} at {COMMIT[:7]}: validation scan AUC {v['scan_auc']}, sensitivity {v['sensitivity']}, "
          f"specificity {v['specificity']} at {calibrator.threshold}; {metrics['minutes_total']} min. Test split not opened. "
          "For Run 2 set OPEN_TEST_SPLIT = True.")
else:
    t = TEST_REPORT['preregistered']
    print(f"OK: {EXPERIMENT} at {COMMIT[:7]}: test scan AUC {t['scan_auc']} {t['scan_auc_ci95']}, sensitivity {t['sensitivity']} "
          f"{t['sensitivity_ci95']}, specificity {t['specificity']} {t['specificity_ci95']}, ECE {t['scan_ece']}; "
          f"{metrics['minutes_total']} min. The test split has now been opened once; do not run this notebook again.")